#### Data Cleaning, Preprocessing and SQL Ingestion
---
Pipeline Stages:
1. Missing value imputation and lifecycle retention
2. Spatial duplicate elimination (geolocation)
3. Postal code zero-padding (string formatting)
4. Timestamp parsing into native datetimes
5. Category translation and city string standardization
6. Domain record validation (payments, non-positive weights, GPS clipping)
7. Clean CSV persistence (`data/cleaned/`)
8. Relational schema creation and bulk loading in MySQL
9. Key constraint enforcement and row count verification


In [1]:
import pandas as pd
from pathlib import Path


In [ ]:
# Set data path and list all CSV files
RAW_DATA_DIR: Path = Path("../data/raw")


### 1. Ingest Raw Datasets


In [3]:
# Load all 9 raw datasets into DataFrames
df_orders: pd.DataFrame = pd.read_csv(RAW_DATA_DIR / "olist_orders_dataset.csv") 
df_customers: pd.DataFrame = pd.read_csv(RAW_DATA_DIR / "olist_customers_dataset.csv")
df_order_items: pd.DataFrame = pd.read_csv(RAW_DATA_DIR / "olist_order_items_dataset.csv")
df_payments: pd.DataFrame = pd.read_csv(RAW_DATA_DIR / "olist_order_payments_dataset.csv")
df_reviews: pd.DataFrame = pd.read_csv(RAW_DATA_DIR / "olist_order_reviews_dataset.csv")
df_products: pd.DataFrame = pd.read_csv(RAW_DATA_DIR / "olist_products_dataset.csv")
df_sellers: pd.DataFrame = pd.read_csv(RAW_DATA_DIR / "olist_sellers_dataset.csv")
df_geolocation: pd.DataFrame = pd.read_csv(RAW_DATA_DIR / "olist_geolocation_dataset.csv")
df_category_translation: pd.DataFrame = pd.read_csv(RAW_DATA_DIR / "product_category_name_translation.csv")


### 2. Handle Missing Values


#### Products Table Imputation


In [4]:
# Fill missing category names with 'unknown' and impute product dimensions with medians
df_products['product_category_name'] = df_products['product_category_name'].fillna('unknown')

median_weight: float = float(df_products['product_weight_g'].median())
print(f"Median weight of all products: {median_weight} grams")
df_products['product_weight_g'] = df_products['product_weight_g'].fillna(median_weight)

median_length: float = float(df_products['product_length_cm'].median())
print(f"Median length of all products: {median_length} cm")
df_products['product_length_cm'] = df_products['product_length_cm'].fillna(median_length)

median_height: float = float(df_products['product_height_cm'].median())
print(f"Median height of all products: {median_height} cm")
df_products['product_height_cm'] = df_products['product_height_cm'].fillna(median_height)

median_width: float = float(df_products['product_width_cm'].median())
print(f"Median width of all products: {median_width} cm")
df_products['product_width_cm'] = df_products['product_width_cm'].fillna(median_width)


Median weight of all products: 700.0 grams
Median length of all products: 25.0 cm
Median height of all products: 13.0 cm
Median width of all products: 20.0 cm


In [5]:
# Audit remaining missing values in df_products
products_missing_values: pd.Series = df_products.isnull().sum()
print("Missing values in each column:")
print(products_missing_values)


Missing values in each column:
product_id                      0
product_category_name           0
product_name_length           610
product_description_length    610
product_photos_qty            610
product_weight_g                0
product_length_cm               0
product_height_cm               0
product_width_cm                0
dtype: int64


> `product_name_length`, `product_description_length`, and `product_photos_qty` have 610 missing values each. These are optional catalog attributes and do not affect relational keys or analytics.


#### Reviews Table Imputation


In [6]:
# Impute optional feedback fields in df_reviews
df_reviews['review_comment_title'] = df_reviews['review_comment_title'].fillna('No Title')
df_reviews['review_comment_message'] = df_reviews['review_comment_message'].fillna('No comment')

review_missing_values: pd.Series = df_reviews.isnull().sum()
print("Missing values in in df_reviews after filling:")
print(review_missing_values)


Missing values in in df_reviews after filling:
review_id                  0
order_id                   0
review_score               0
review_comment_title       0
review_comment_message     0
review_creation_date       0
review_answer_timestamp    0
dtype: int64


In [7]:
# Verify null values in df_orders
orders_missing_values: pd.Series = df_orders.isnull().sum()
print("Missing values in df_orders:")
print(orders_missing_values)


Missing values in df_orders:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


> Unpopulated delivery timestamps in `df_orders` represent legitimate operational lifecycle states (canceled or in-transit shipments; Missing Not At Random) and are preserved without imputation.


### 3. Remove Duplicate Records


In [8]:
# Remove exact duplicate coordinate readings from df_geolocation
geo_rows_before: int = len(df_geolocation)
df_geolocation = df_geolocation.drop_duplicates()
geo_rows_after: int = len(df_geolocation)
geo_duplicates_removed: int = geo_rows_before - geo_rows_after

print(f"Total rows before removing duplicates : {geo_rows_before}")
print(f"Total duplicate rows removed          : {geo_duplicates_removed}")
print(f"Total rows after removing duplicates  : {geo_rows_after}")

is_remaining_duplicates: int = int(df_geolocation.duplicated().sum())
print(f"Total duplicate rows remaining after renaming columns: {is_remaining_duplicates}")


Total rows before removing duplicates : 1000163
Total duplicate rows removed          : 261831
Total rows after removing duplicates  : 738332
Total duplicate rows remaining after renaming columns: 0


### 4. Correct Data Types (Postal Codes)


In [9]:
# Pad postal code prefixes to 5-digit strings to preserve leading zeros
df_customers['customer_zip_code_prefix'] = df_customers['customer_zip_code_prefix'].astype(str).str.zfill(5)
df_sellers['seller_zip_code_prefix'] = df_sellers['seller_zip_code_prefix'].astype(str).str.zfill(5)
df_geolocation['geolocation_zip_code_prefix'] = df_geolocation['geolocation_zip_code_prefix'].astype(str).str.zfill(5)


In [10]:
# Verify updated postal code formats and sample values
print("Customer zip code column datatype    :", df_customers['customer_zip_code_prefix'].dtype)
print("Seller zip code column datatype      :", df_sellers['seller_zip_code_prefix'].dtype)
print("Geolocation zip code column datatype :", df_geolocation['geolocation_zip_code_prefix'].dtype)

print("Customer zip code sample values    :", df_customers['customer_zip_code_prefix'].head())
print("Seller zip code sample values      :", df_sellers['seller_zip_code_prefix'].head())
print("Geolocation zip code sample values :", df_geolocation['geolocation_zip_code_prefix'].head())


Customer zip code column datatype    : str
Seller zip code column datatype      : str
Geolocation zip code column datatype : str
Customer zip code sample values    : 0    14409
1    09790
2    01151
3    08775
4    13056
Name: customer_zip_code_prefix, dtype: str
Seller zip code sample values      : 0    13023
1    13844
2    20031
3    04195
4    12914
Name: seller_zip_code_prefix, dtype: str
Geolocation zip code sample values : 0    01037
1    01046
2    01046
3    01041
4    01035
Name: geolocation_zip_code_prefix, dtype: str


### 5. Format Date and Time Columns


Parse 8 timestamp columns from string (`object`) to `datetime64[ns]` across orders, order items, and reviews.


In [11]:
# Parse timestamp columns in df_orders to datetime
df_orders['order_purchase_timestamp'] = pd.to_datetime(df_orders['order_purchase_timestamp'])
df_orders['order_approved_at'] = pd.to_datetime(df_orders['order_approved_at'])
df_orders['order_delivered_carrier_date'] = pd.to_datetime(df_orders['order_delivered_carrier_date'])
df_orders['order_delivered_customer_date'] = pd.to_datetime(df_orders['order_delivered_customer_date'])
df_orders['order_estimated_delivery_date'] = pd.to_datetime(df_orders['order_estimated_delivery_date'])

print("Data types in df_orders:")
print(df_orders[['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']].dtypes)


Data types in df_orders:
order_purchase_timestamp         datetime64[us]
order_approved_at                datetime64[us]
order_delivered_carrier_date     datetime64[us]
order_delivered_customer_date    datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [12]:
# Parse shipping limit timestamp in df_order_items
df_order_items['shipping_limit_date'] = pd.to_datetime(df_order_items['shipping_limit_date'])
print("Data type in df_order_items:", df_order_items['shipping_limit_date'].dtypes)


Data type in df_order_items: datetime64[us]


In [13]:
# Parse feedback timestamps in df_reviews
df_reviews['review_creation_date'] = pd.to_datetime(df_reviews['review_creation_date'])
df_reviews['review_answer_timestamp'] = pd.to_datetime(df_reviews['review_answer_timestamp'])
print("Data types in df_reviews:")
print(df_reviews[['review_creation_date', 'review_answer_timestamp']].dtypes)


Data types in df_reviews:
review_creation_date       datetime64[us]
review_answer_timestamp    datetime64[us]
dtype: object


### 6. Standardize Categorical Data and Text


In [14]:
# Merge English product categories and normalize city names
df_products = pd.merge(
    df_products,
    df_category_translation,
    on='product_category_name',
    how='left'
)
df_products['product_category_name_english'] = df_products['product_category_name_english'].fillna(df_products['product_category_name'])

df_customers['customer_city'] = df_customers['customer_city'].str.lower().str.strip()
df_sellers['seller_city'] = df_sellers['seller_city'].str.lower().str.strip()
df_geolocation['geolocation_city'] = df_geolocation['geolocation_city'].str.lower().str.strip()


In [15]:
# Verify category translation samples and check for unmapped values
sample_translations: pd.DataFrame = df_products[['product_category_name', 'product_category_name_english']].head(5)
print("Sample Category Translations:")
print(sample_translations)

english_nulls: int = int(df_products['product_category_name_english'].isna().sum())
print("\nRemaining nulls in English category column:", english_nulls)


Sample Category Translations:
   product_category_name product_category_name_english
0             perfumaria                     perfumery
1                  artes                           art
2          esporte_lazer                sports_leisure
3                  bebes                          baby
4  utilidades_domesticas                    housewares

Remaining nulls in English category column: 0


### 7. Handle Invalid Records and Spatial Outliers


In [16]:
# Remediate payment types, impute non-positive weights, and clip GPS coordinates
df_payments['payment_type'] = df_payments['payment_type'].replace('not_defined', 'other')

median_prod_weight: float = float(df_products['product_weight_g'].median())
df_products.loc[df_products['product_weight_g'] <= 0, 'product_weight_g'] = median_prod_weight

valid_brazil_lat: pd.Series = (df_geolocation['geolocation_lat'] >= -34.0) & (df_geolocation['geolocation_lat'] <= 5.5)
valid_brazil_lng: pd.Series = (df_geolocation['geolocation_lng'] >= -74.0) & (df_geolocation['geolocation_lng'] <= -34.0)
df_geolocation = df_geolocation[valid_brazil_lat & valid_brazil_lng]


In [17]:
# Verify remediation results
not_defined_count: int = int((df_payments['payment_type'] == 'not_defined').sum())
print("Count of 'not_defined' in payment_type :", not_defined_count)

min_product_weight: float = float(df_products['product_weight_g'].min())
print("Minimum product weight in df_products :", min_product_weight, "grams")

print("\nCleaned GPS Coordinates Summary:")
print(df_geolocation[['geolocation_lat', 'geolocation_lng']].describe().round(2))


Count of 'not_defined' in payment_type : 0
Minimum product weight in df_products : 2.0 grams

Cleaned GPS Coordinates Summary:
       geolocation_lat  geolocation_lng
count        738299.00        738299.00
mean            -21.00           -46.46
std               5.88             4.38
min             -33.69           -72.93
25%             -23.60           -48.87
50%             -22.87           -46.65
75%             -19.92           -43.84
max               4.48           -34.79


### 8. Export Cleaned Datasets to CSV


In [18]:
# Export cleaned tables to data/cleaned/
CLEAN_DATA_DIR: Path = Path("../data/cleaned")

df_orders.to_csv(CLEAN_DATA_DIR / "olist_orders_cleaned.csv", index=False)
df_customers.to_csv(CLEAN_DATA_DIR / "olist_customers_cleaned.csv", index=False)
df_order_items.to_csv(CLEAN_DATA_DIR / "olist_order_items_cleaned.csv", index=False)
df_payments.to_csv(CLEAN_DATA_DIR / "olist_order_payments_cleaned.csv", index=False)
df_reviews.to_csv(CLEAN_DATA_DIR / "olist_order_reviews_cleaned.csv", index=False)
df_products.to_csv(CLEAN_DATA_DIR / "olist_products_cleaned.csv", index=False)
df_sellers.to_csv(CLEAN_DATA_DIR / "olist_sellers_cleaned.csv", index=False)
df_geolocation.to_csv(CLEAN_DATA_DIR / "olist_geolocation_cleaned.csv", index=False)


> Product category translation is incorporated directly as `product_category_name_english` in `olist_products_cleaned.csv`.


In [19]:
# Verify exported cleaned files
cleaned_csv_files: list[Path] = []
for file in CLEAN_DATA_DIR.glob("*.csv"):
    cleaned_csv_files.append(file)
cleaned_csv_files.sort()

print("Total cleaned CSV files found :", len(cleaned_csv_files))
for file in cleaned_csv_files:
    print(file.name)


Total cleaned CSV files found : 8
olist_customers_cleaned.csv
olist_geolocation_cleaned.csv
olist_order_items_cleaned.csv
olist_order_payments_cleaned.csv
olist_order_reviews_cleaned.csv
olist_orders_cleaned.csv
olist_products_cleaned.csv
olist_sellers_cleaned.csv


### 9. Relational Database Initialization


In [ ]:
# Connect to MySQL server and initialize ecommerce_db
from sqlalchemy import create_engine, text

DB_USER: str = "root"
DB_PASSWORD: str = "1234"
DB_HOST: str = "localhost"
DB_PORT: str = "3306"
DB_NAME: str = "ecommerce_db"

server_url: str = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}"
server_engine = create_engine(server_url)

with server_engine.connect() as connection:
    connection.execute(text(f"CREATE DATABASE IF NOT EXISTS {DB_NAME};"))
    connection.commit()

db_url: str = f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
db_engine = create_engine(db_url)

with db_engine.connect() as connection:
    result = connection.execute(text("SELECT DATABASE();"))
    current_db: str = str(result.scalar())

print(f"Successfully connected to MySQL! Active Database: '{current_db}'")


Successfully connected to MySQL! Active Database: 'ecommerce_db'


### 10. Create Relational Tables (ER Diagram Alignment)


In [ ]:
# Define SQL DDL table creation statements matching our ER diagram
table_queries: dict[str, str] = {
    # 1. Customers Dimension Table
    "customers": """
        CREATE TABLE IF NOT EXISTS customers (
            customer_id VARCHAR(32) PRIMARY KEY,
            customer_unique_id VARCHAR(32) NOT NULL,
            customer_zip_code_prefix VARCHAR(5) NOT NULL,
            customer_city VARCHAR(100),
            customer_state VARCHAR(2)
        );
    """,

    # 2. Sellers Dimension Table
    "sellers": """
        CREATE TABLE IF NOT EXISTS sellers (
            seller_id VARCHAR(32) PRIMARY KEY,
            seller_zip_code_prefix VARCHAR(5) NOT NULL,
            seller_city VARCHAR(100),
            seller_state VARCHAR(2)
        );
    """,

    # 3. Products Dimension Table
    "products": """
        CREATE TABLE IF NOT EXISTS products (
            product_id VARCHAR(32) PRIMARY KEY,
            product_category_name VARCHAR(100),
            product_name_length FLOAT,
            product_description_length FLOAT,
            product_photos_qty FLOAT,
            product_weight_g FLOAT,
            product_length_cm FLOAT,
            product_height_cm FLOAT,
            product_width_cm FLOAT,
            product_category_name_english VARCHAR(100)
        );
    """,

    # 4. Orders Core Fact Table
    "orders": """
        CREATE TABLE IF NOT EXISTS orders (
            order_id VARCHAR(32) PRIMARY KEY,
            customer_id VARCHAR(32) NOT NULL,
            order_status VARCHAR(20) NOT NULL,
            order_purchase_timestamp DATETIME NOT NULL,
            order_approved_at DATETIME,
            order_delivered_carrier_date DATETIME,
            order_delivered_customer_date DATETIME,
            order_estimated_delivery_date DATETIME
        );
    """,

    # 5. Order Items Fact Table (Composite PK: order_id + order_item_id)
    # composite primary key: order_id + order_item_id, Because each order can have multiple items
    "order_items": """
        CREATE TABLE IF NOT EXISTS order_items (
            order_id VARCHAR(32) NOT NULL,
            order_item_id INT NOT NULL,
            product_id VARCHAR(32) NOT NULL,
            seller_id VARCHAR(32) NOT NULL,
            shipping_limit_date DATETIME NOT NULL,
            price DECIMAL(10, 2) NOT NULL,
            freight_value DECIMAL(10, 2) NOT NULL,
            PRIMARY KEY (order_id, order_item_id)
        );
    """,

    # 6. Order Payments Fact Table (Composite PK: order_id + payment_sequential)
    # order_id is a foreign key referencing orders(order_id)
    # Composite primary key: order_id + payment_sequential, Because each order can have multiple payments
    "order_payments": """
        CREATE TABLE IF NOT EXISTS order_payments (
            order_id VARCHAR(32) NOT NULL,
            payment_sequential INT NOT NULL,
            payment_type VARCHAR(20) NOT NULL,
            payment_installments INT NOT NULL,
            payment_value DECIMAL(10, 2) NOT NULL,
            PRIMARY KEY (order_id, payment_sequential)
        );
    """,

    # 7. Order Reviews Fact Table (Composite PK: review_id + order_id)
    # order_id is a foreign key referencing orders(order_id)
    # Composite primary key: review_id + order_id, Because each order can have multiple reviews
    "order_reviews": """
        CREATE TABLE IF NOT EXISTS order_reviews (
            review_id VARCHAR(32) NOT NULL,
            order_id VARCHAR(32) NOT NULL,
            review_score INT NOT NULL,
            review_comment_title VARCHAR(255),
            review_comment_message TEXT,
            review_creation_date DATETIME NOT NULL,
            review_answer_timestamp DATETIME NOT NULL,
            PRIMARY KEY (review_id, order_id)
        );
    """,

    # 8. Geolocation Spatial Table
    "geolocation": """
        CREATE TABLE IF NOT EXISTS geolocation (
            geolocation_zip_code_prefix VARCHAR(5) NOT NULL,
            geolocation_lat DECIMAL(10, 6) NOT NULL,
            geolocation_lng DECIMAL(10, 6) NOT NULL,
            geolocation_city VARCHAR(100),
            geolocation_state VARCHAR(2)
        );
    """
}

with db_engine.connect() as connection:
    for table_name, query in table_queries.items():
        connection.execute(text(query))
    connection.commit()

with db_engine.connect() as connection:
    result = connection.execute(text("SHOW TABLES;"))
    created_tables: list[str] = [row[0] for row in result.fetchall()]

print(f"Total tables created in 'ecommerce_db': {len(created_tables)}\n")
for idx, tbl in enumerate(created_tables, start=1):
    print(f"{idx}. {tbl}")


Total tables created in 'ecommerce_db': 8

1. customers
2. geolocation
3. order_items
4. order_payments
5. order_reviews
6. orders
7. products
8. sellers


### 11. Bulk Load Cleaned Datasets into MySQL


In [ ]:
# Map cleaned DataFrames to their corresponding MySQL table names
# Note: Parent / Dimension tables first, followed by Child / Fact tables
data_to_load: dict[str, pd.DataFrame] = {
    # 1. Customers table
    "customers": df_customers,

    # 2. Sellers table
    "sellers": df_sellers,

    # 3. Products table
    "products": df_products,

    # 4. Geolocation table
    "geolocation": df_geolocation,

    # 5. Orders table
    "orders": df_orders,

    # 6. Order Items table
    "order_items": df_order_items,

    # 7. Order Payments table
    "order_payments": df_payments,

    # 8. Order Reviews table
    "order_reviews": df_reviews
}

# Iterate through each table and stream data into MySQL using chunksize
for table_name, df_table in data_to_load.items():
    print(f"Loading '{table_name}', {len(df_table):,} rows")

    df_table.to_sql(
        name=table_name,
        con=db_engine,
        if_exists="append",
        index=False,
        chunksize=10000
    )

    print(f" '{table_name}' loaded successfully\n")


Loading 'customers' (99,441 rows)...
 'customers' loaded successfully

Loading 'sellers' (3,095 rows)...
 'sellers' loaded successfully

Loading 'products' (32,951 rows)...
 'products' loaded successfully

Loading 'geolocation' (738,299 rows)...
 'geolocation' loaded successfully

Loading 'orders' (99,441 rows)...
 'orders' loaded successfully

Loading 'order_items' (112,650 rows)...
 'order_items' loaded successfully

Loading 'order_payments' (103,886 rows)...
 'order_payments' loaded successfully

Loading 'order_reviews' (99,224 rows)...
 'order_reviews' loaded successfully

All 8 tables successfully loaded into MySQL 'ecommerce_db'!


### 12. Apply Foreign Key Constraints


In [ ]:
# Define Foreign Key constraints to link tables based on ER diagram
foreign_key_queries: list[str] = [
    # 1. Link orders to customers
    "ALTER TABLE orders ADD CONSTRAINT fk_orders_customer FOREIGN KEY (customer_id) REFERENCES customers(customer_id);",

    # 2. Link order_items to orders
    "ALTER TABLE order_items ADD CONSTRAINT fk_items_order FOREIGN KEY (order_id) REFERENCES orders(order_id);",

    # 3. Link order_items to products
    "ALTER TABLE order_items ADD CONSTRAINT fk_items_product FOREIGN KEY (product_id) REFERENCES products(product_id);",

    # 4. Link order_items to sellers
    "ALTER TABLE order_items ADD CONSTRAINT fk_items_seller FOREIGN KEY (seller_id) REFERENCES sellers(seller_id);",

    # 5. Link order_payments to orders
    "ALTER TABLE order_payments ADD CONSTRAINT fk_payments_order FOREIGN KEY (order_id) REFERENCES orders(order_id);",

    # 6. Link order_reviews to orders
    "ALTER TABLE order_reviews ADD CONSTRAINT fk_reviews_order FOREIGN KEY (order_id) REFERENCES orders(order_id);"
]

with db_engine.connect() as connection:
    for fk_query in foreign_key_queries:
        connection.execute(text(fk_query))
    connection.commit()

print("All 6 Foreign Key constraints successfully applied")


All 6 Foreign Key constraints successfully applied! Relational integrity enforced.


### 13. Verify Database Row Counts against DataFrames


In [27]:
# Verify row counts across all 8 SQL tables against in-memory DataFrames
verification_results: list[dict[str, any]] = []

expected_counts: dict[str, int] = {
    "customers": len(df_customers),
    "sellers": len(df_sellers),
    "products": len(df_products),
    "geolocation": len(df_geolocation),
    "orders": len(df_orders),
    "order_items": len(df_order_items),
    "order_payments": len(df_payments),
    "order_reviews": len(df_reviews)
}

with db_engine.connect() as connection:
    for table_name, expected_row_count in expected_counts.items():
        result = connection.execute(text(f"SELECT COUNT(*) FROM {table_name};"))
        sql_row_count: int = int(result.scalar())
        is_matched: bool = (sql_row_count == expected_row_count)
        status: str = "Match" if is_matched else "Mismatch"
        
        verification_results.append({
            "Table Name": table_name,
            "Pandas Rows": expected_row_count,
            "SQL Rows": sql_row_count,
            "Status": status
        })

df_verification: pd.DataFrame = pd.DataFrame(verification_results)
print("SQL Database Verification Report:")
print(df_verification.to_string(index=False))


SQL Database Verification Report:
    Table Name  Pandas Rows  SQL Rows Status
     customers        99441     99441  Match
       sellers         3095      3095  Match
      products        32951     32951  Match
   geolocation       738299    738299  Match
        orders        99441     99441  Match
   order_items       112650    112650  Match
order_payments       103886    103886  Match
 order_reviews        99224     99224  Match
